# Convert MIDI to piano roll images (CNN input)

Turns each filtered MIDI file into a piano roll matrix (pitch x time) that the CNN will treat like an image.

Settings:
- fs=4 (4 time steps per second). Tried fs=10 and fs=100 first but the files blow up in size fast, some pieces are 20+ minutes long. fs=4 keeps things manageable and piano roll images don't need crazy time resolution anyway.
- Only keeping pitches 21-108 (that's the actual 88 key piano range), no point keeping the full 0-127 midi range when most of it is empty.
- Saving as uint8 (velocity fits in 0-127 anyway) instead of float, cuts file size by 4x vs float32.

Input: `data/raw/<composer>/*.mid`

Output: `data/processed/cnn/<composer>/<file>.npy` + manifest csv

In [1]:
import warnings
import pretty_midi
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings("ignore", category=RuntimeWarning)

FS = 4
PITCH_LOW = 21
PITCH_HIGH = 109  # exclusive, so this covers 21-108

In [2]:
RAW_DIR = Path("../data/raw")
OUT_DIR = Path("../data/processed/cnn")
OUT_DIR.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(RAW_DIR / "manifest.csv")
manifest.head()

,composer,filename,source_path
0,bach,AveMaria.mid,Bach\AveMaria.mid
1,bach,01_Menuet.mid,Bach\Bwv ''Little Notebook for Anna Magdalena ...
2,bach,02_Menuet.mid,Bach\Bwv ''Little Notebook for Anna Magdalena ...
3,bach,03_Menuet.mid,Bach\Bwv ''Little Notebook for Anna Magdalena ...
4,bach,04_Menuet.mid,Bach\Bwv ''Little Notebook for Anna Magdalena ...


In [3]:
def midi_to_piano_roll(path):
    pm = pretty_midi.PrettyMIDI(str(path))
    roll = pm.get_piano_roll(fs=FS)  # shape (128, time)

    if roll.shape[1] == 0:
        raise ValueError("empty piano roll")

    roll = roll[PITCH_LOW:PITCH_HIGH, :]  # trim to 88 key piano range
    roll = np.clip(roll, 0, 127).astype(np.uint8)
    return roll

In [4]:
# test on one file, make sure shape looks right
test_row = manifest.iloc[0]
test_path = RAW_DIR / test_row["composer"] / test_row["filename"]
roll = midi_to_piano_roll(test_path)
print(test_row["filename"], "-> roll shape:", roll.shape, "dtype:", roll.dtype, "max:", roll.max())

AveMaria.mid -> roll shape: (88, 328) dtype: uint8 max: 85


In [5]:
results = []
failed = []

for i, row in manifest.iterrows():
    composer = row["composer"]
    filename = row["filename"]
    src = RAW_DIR / composer / filename

    try:
        roll = midi_to_piano_roll(src)
    except Exception as e:
        failed.append({"composer": composer, "filename": filename, "error": str(e)})
        continue

    comp_dir = OUT_DIR / composer
    comp_dir.mkdir(exist_ok=True)
    out_path = comp_dir / (Path(filename).stem + ".npy")
    np.save(out_path, roll)

    results.append({
        "composer": composer,
        "filename": filename,
        "npy_path": str(out_path.relative_to(OUT_DIR.parent.parent)),
        "time_steps": roll.shape[1],
        "file_kb": round(out_path.stat().st_size / 1024, 1),
    })

    if i % 300 == 0:
        print(f"...{i}/{len(manifest)} done")

print("finished. ok:", len(results), "failed:", len(failed))

...0/1637 done


...300/1637 done


...600/1637 done


...900/1637 done


...1200/1637 done


...1500/1637 done


finished. ok: 1635 failed: 2


In [6]:
roll_df = pd.DataFrame(results)
roll_df.to_csv("../data/processed/cnn_manifest.csv", index=False)

fail_df = pd.DataFrame(failed)
fail_df.to_csv("../data/processed/cnn_failed.csv", index=False)

roll_df.groupby("composer")["time_steps"].agg(["count", "mean", "min", "max"])

,count,mean,min,max
composer,,,,
bach,1024,619.552734,70,20672
beethoven,219,1970.657534,87,9999
chopin,136,881.132353,93,5410
mozart,256,1597.023438,104,5912


In [7]:
print("total size of data/processed/cnn:")
total_kb = roll_df["file_kb"].sum()
print(f"{total_kb/1024:.1f} MB")

total size of data/processed/cnn:
134.0 MB


In [8]:
fail_df

,composer,filename,error
0,beethoven,Anhang_14-3.mid,Could not decode key with 3 flats and mode 255
1,mozart,K281_Piano_Sonata_n03_3mov.mid,Could not decode key with 2 flats and mode 2


## Notes

- same 2 files failed here as in the LSTM notebook (corrupt key signature metadata), consistent with what we saw before so that's expected, not a new bug
- time_steps varies a lot same as the LSTM sequence lengths did, since it's the same underlying files just represented differently. CNN model step will need to pick a fixed window size or resize these
- pitch axis is fixed at 88 (the piano range) for every file so that part's consistent already, that's the part that actually matters for a CNN input shape